# Pomodoro adaptativo com RL: simulador, dataset e agente
Execute na ordem. Todos os parâmetros do simulador, da recompensa e do agente ficam em `config/padrao.json` (ou em `OVERRIDES` abaixo). Os dados são **simulados**: servem para testar o pipeline e estudar sensibilidade, não são medições de alunos reais.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'src/treino.py').exists(), 'Ajuste ROOT para a pasta do projeto'
sys.path.insert(0, str(ROOT/'src'))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(ROOT/'tests')], check=True)

## Parâmetros de execução
`N_CPUS` = execuções simultâneas (1 CPU cada). `OVERRIDES` sobrescreve qualquer parâmetro da config sem editar o arquivo (mesma sintaxe do `--set`).

In [ ]:
N_CPUS = 4            # <-- ESCOLHA quantas CPUs usar
RAPIDO = True         # True: poucos episódios (teste de fumaça). Use False para a grade completa
OVERRIDES = [         # exemplos: 'simulador.ruido_autorrelato=0', 'recompensa.penalidade=20'
]
CONFIG = ROOT/'config/padrao.json'
TOTAL_CPUS = os.cpu_count() or 1
N_CPUS = max(1, min(N_CPUS, TOTAL_CPUS))
print(f'CPUs disponíveis: {TOTAL_CPUS} | usando: {N_CPUS}')
from config import montar
cfg = montar(CONFIG, sets=OVERRIDES, rapido=RAPIDO)
print(json.dumps({k: cfg[k] for k in ['simulador','recompensa','agente','treino']}, indent=1, ensure_ascii=False))

## 1. Simulador: como cada ciclo afeta a fadiga
Compara os três ciclos a partir do mesmo estado (fadiga latente 0,2; sem ruído de autorrelato).

In [ ]:
import numpy as np
from simulador import SimuladorPomodoro
from config import ACOES
sem_ruido = montar(CONFIG, sets=OVERRIDES+['simulador.ruido_autorrelato=0'])
for u, nome in enumerate(['longe','semana','vespera']):
    linha = []
    for a, nm in enumerate(ACOES):
        e = SimuladorPomodoro(sem_ruido, np.random.default_rng(0)); e.reset(urgencia=u, fadiga=0.2)
        _, r, _, i = e.step(a); linha.append(f"{nm}: F=0.20->{i['fadiga']:.2f}, min_ef={i['minutos_efetivos']:.1f}")
    print(nome.ljust(8), ' | '.join(linha))

## 2. Dataset simulado (política aleatória)
Gera transições (s, a, r, s') para análise ou RL offline. Cada shard é um processo; a barra conta episódios.

In [ ]:
import executar_grade
from paralelo import run_jobs
DS = ROOT/'data/dataset_simulado'
jobs, _ = executar_grade.jobs_dataset(CONFIG, DS, shards=N_CPUS, episodios=4000 if RAPIDO else 80000, sets=OVERRIDES, rapido=RAPIDO)
run_jobs(jobs, workers=N_CPUS, desc='Dataset')
caminho, n = executar_grade.consolidar_dataset(DS)
import pandas as pd
df = pd.read_csv(caminho); print(n, 'transições'); display(df.head())
display(df.groupby(['cansaco','acao']).cansaco_alto_seguinte.mean().unstack().round(2))

## 3. Grade de agentes e baselines
Q-learning e bandit × variantes de recompensa × penalidades × sementes, mais as políticas fixas, a regra e a aleatória. Pastas concluídas são puladas (retomável).

In [ ]:
GRADE = ROOT/'experiments/grade_rapida' if RAPIDO else ROOT/'experiments/grade'
jobs, total = executar_grade.jobs_grade(CONFIG, GRADE, sets=OVERRIDES, rapido=RAPIDO)
print(f'{len(jobs)} de {total} execuções pendentes, {sum(j.units for j in jobs)} episódios, {N_CPUS} em paralelo')
RUN_GRID = True
if RUN_GRID:
    run_jobs(jobs, workers=N_CPUS, desc='Grade')

## 4. Resultados
Compare somente métricas independentes da recompensa (minutos efetivos e frequência de cansaço alto). A recompensa de variantes diferentes não é comparável.

In [ ]:
import analise
df_exec = analise.carregar(GRADE); tab = analise.tabela(df_exec)
display(tab.round(3)); analise.graficos(GRADE, df_exec, tab)
from IPython.display import Image
display(Image(filename=str(GRADE/'pareto.png'))); display(Image(filename=str(GRADE/'curvas.png')))

In [ ]:
# Política aprendida em uma execução (linhas: cansaço 1-5; colunas: longe | semana | vespera)
analise.mostrar_politica(GRADE, 'qlearning_eficaz_p10_s11')

## 5. Próximos passos sugeridos
Sensibilidade do ruído (`simulador.ruido_autorrelato=0` vs 0.35), do limiar de cansaço alto, da razão +1/penalidade; RL offline sobre o dataset; coleta com voluntários reais (app que sorteia o ciclo e pergunta o cansaço 1-5). Veja o README.